___

TOPIC MODELLING
___

# Latent Dirichlet Allocation

## Data

We will be using articles from NPR (National Public Radio), obtained from their website [www.npr.org](http://www.npr.org)

In [1]:
import pandas as pd

In [2]:
url = 'https://raw.githubusercontent.com/nluninja/text-mining-dataviz-aa2526/refs/heads/main/06-Topic_Modeling/npr.csv'

In [3]:
npr = pd.read_csv(url)

In [4]:
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


Notice how we don't have the topic of the articles! Let's use LDA to attempt to figure out clusters of the articles.

In [5]:
npr.shape

(11992, 1)

## Preprocessing

In [6]:
from sklearn.feature_extraction.text import CountVectorizer

**`max_df`**` : float in range [0.0, 1.0] or int, default=1.0`<br>
When building the vocabulary ignore terms that have a document frequency strictly higher than the given threshold (corpus-specific stop words). If float, the parameter represents a proportion of documents, integer absolute counts. This parameter is ignored if vocabulary is not None.

**`min_df`**` : float in range [0.0, 1.0] or int, default=1`<br>
When building the vocabulary ignore terms that have a document frequency strictly lower than the given threshold. This value is also called cut-off in the literature. If float, the parameter represents a proportion of documents, integer absolute counts. This parameter is ignored if vocabulary is not None.

In [7]:
cv = CountVectorizer(max_df=0.95, min_df=0.2, stop_words='english')

In [8]:
dtm = cv.fit_transform(npr['Article'])

In [9]:
dtm

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 349867 stored elements and shape (11992, 96)>

## LDA

In [10]:
from sklearn.decomposition import LatentDirichletAllocation

In [11]:
LDA = LatentDirichletAllocation(n_components=7,max_iter=50,random_state=42)

In [12]:
# This can take awhile, we're dealing with a large amount of documents!
LDA.fit(dtm)

LatentDirichletAllocation(max_iter=50, n_components=7, random_state=42)

## Showing Stored Words

In [13]:
len(cv.get_feature_names_out())

96

### Showing Top Words Per Topic

In [14]:
len(LDA.components_)

7

In [15]:
LDA.components_

array([[3.10749994e+01, 2.39317109e+01, 1.97880251e+02, 1.43454304e-01,
        1.48235707e+01, 1.43453426e-01, 2.32973529e+01, 1.08155998e+01,
        1.08457060e+02, 1.43444876e-01, 1.65515728e-01, 4.61383446e+02,
        1.47961332e+02, 6.41312208e+01, 1.31140436e+02, 2.36680691e+02,
        4.92299621e+01, 1.72581410e+01, 7.56883598e+01, 1.43440238e-01,
        3.06075894e+01, 2.28623091e+02, 1.87607058e+02, 2.91707756e+01,
        2.91282710e+01, 6.27384533e+01, 1.02736004e+02, 1.43426251e-01,
        2.02952192e+02, 1.83116491e+02, 9.39690940e+00, 1.43488259e-01,
        2.47931760e+02, 1.32774327e+01, 9.69986529e+01, 3.14208384e+02,
        2.51850718e+02, 7.91366581e+01, 2.29182824e+02, 1.43452801e-01,
        1.43409202e-01, 1.33923932e+00, 1.43291573e-01, 5.04846133e+01,
        5.95641336e+01, 8.29494406e+00, 5.35147971e+01, 5.25511098e+01,
        3.41941966e+01, 2.80753136e+02, 1.17014777e+02, 5.43726034e+02,
        1.80027762e+02, 1.38376735e+03, 1.43385915e-01, 1.274139

In [16]:
len(LDA.components_[0])

96

In [17]:
single_topic = LDA.components_[0]

In [18]:
# Returns the indices that would sort this array.
single_topic.argsort()

array([80, 93, 42, 63, 68, 78, 54, 76, 79, 90, 72, 40, 71, 91, 27, 19,  9,
       89, 39,  5,  3, 31, 65, 10, 58, 92, 41, 74, 57, 86, 45, 30,  7, 33,
       85, 81,  4, 17, 60,  6,  1, 82, 73, 24, 23, 20,  0, 84, 67, 48, 16,
       43, 47, 46, 44, 25, 13, 66, 18, 77, 37, 59, 95, 56, 34, 75, 26,  8,
       50, 87, 55, 14, 12, 52, 64, 29, 22,  2, 28, 61, 21, 38, 83, 15, 88,
       32, 36, 49, 35, 11, 94, 51, 62, 53, 70, 69])

In [19]:
# Word least representative of this topic
single_topic[80]

np.float64(0.143157297330522)

In [20]:
# Word most representative of this topic
single_topic[69]

np.float64(10081.140841839071)

In [21]:
# Top 10 words for this topic:
single_topic.argsort()[-10:]

array([36, 49, 35, 11, 94, 51, 62, 53, 70, 69])

In [22]:
top_word_indices = single_topic.argsort()[-10:]

In [23]:
for index in top_word_indices:
    print(cv.get_feature_names_out()[index])

house
make
home
case
year
national
public
new
states
state


These look like business articles perhaps... Let's confirm by using .transform() on our vectorized articles to attach a label number. But first, let's view all the 10 topics found.

In [24]:
for index,topic in enumerate(LDA.components_):
    print(f'THE TOP 15 WORDS FOR TOPIC #{index}')
    print([cv.get_feature_names_out()[i] for i in topic.argsort()[-15:]])
    print('\n')

THE TOP 15 WORDS FOR TOPIC #0
['just', 'use', 'country', 'week', 'government', 'house', 'make', 'home', 'case', 'year', 'national', 'public', 'new', 'states', 'state']


THE TOP 15 WORDS FOR TOPIC #1
['10', 'university', 'national', 'public', 'change', 'according', 'american', 'years', 'country', 'world', '000', 'new', 'year', 'people', 'percent']


THE TOP 15 WORDS FOR TOPIC #2
['say', 'make', 'want', 'things', 'lot', 've', 'way', 'going', 'really', 'don', 'know', 'just', 'think', 'like', 'people']


THE TOP 15 WORDS FOR TOPIC #3
['called', 'group', 'did', 'time', 'public', 'president', 'case', 'say', 'according', 'news', 'government', 'npr', 'told', 'people', 'said']


THE TOP 15 WORDS FOR TOPIC #4
['used', 'make', 'going', 'work', 'use', 'new', 'help', 'say', 'university', 'don', 'years', 'just', 'like', 'people', 'says']


THE TOP 15 WORDS FOR TOPIC #5
['came', 'did', 'year', 'way', 'home', 'story', 'world', 'family', 'day', 'just', 'life', 'years', 'like', 'new', 'time']


THE TOP

### Attaching Discovered Topic Labels to Original Articles

In [25]:
dtm

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 349867 stored elements and shape (11992, 96)>

In [26]:
dtm.shape

(11992, 96)

In [27]:
len(npr)

11992

In [28]:
topic_results = LDA.transform(dtm)

In [29]:
topic_results.shape

(11992, 7)

In [30]:
topic_results[0]

array([0.00150734, 0.00151148, 0.00151227, 0.0015112 , 0.00150877,
       0.32758589, 0.66486306])

In [31]:
topic_results[0].round(2)

array([0.  , 0.  , 0.  , 0.  , 0.  , 0.33, 0.66])

In [32]:
topic_results[0].argmax()

np.int64(6)

This means that our model thinks that the first article belongs to topic #7.

### Combining with Original Data

In [33]:
npr.head()

,Article
0,"In the Washington of 2016, even when the polic..."
1,Donald Trump has used Twitter — his prefe...
2,Donald Trump is unabashedly praising Russian...
3,"Updated at 2:50 p. m. ET, Russian President Vl..."
4,"From photography, illustration and video, to d..."


In [34]:
topic_results.argmax(axis=1)

array([6, 6, 6, ..., 5, 2, 4])

In [35]:
npr['Topic'] = topic_results.argmax(axis=1)

In [36]:
npr.head(10)

,Article,Topic
0,"In the Washington of 2016, even when the polic...",6
1,Donald Trump has used Twitter — his prefe...,6
2,Donald Trump is unabashedly praising Russian...,6
3,"Updated at 2:50 p. m. ET, Russian President Vl...",6
4,"From photography, illustration and video, to d...",1
5,I did not want to join yoga class. I hated tho...,2
6,With a who has publicly supported the debunk...,1
7,"I was standing by the airport exit, debating w...",5
8,"If movies were trying to be more realistic, pe...",4
9,"Eighteen years ago, on New Year’s Eve, David F...",5


## Visualizing LDA

**pyLDAvis** is designed to help users interpret the topics in a topic model that has been fit to a corpus of text data. The package extracts information from a fitted LDA topic model to inform an interactive web-based visualization.

it's a Python library for interactive topic model visualization, port of the fabulous R package by Carson Sievert and Kenny Shirley.

In [37]:
!pip install pyldavis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.5 MB/s eta 0:00:00


pyLDAvis now also supports LDA application from scikit-learn, but also gensim, that is an alternative library for LDA

In [38]:
import pyLDAvis
import pyLDAvis.lda_model
pyLDAvis.enable_notebook()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [39]:
pyLDAvis.lda_model.prepare(LDA, dtm, cv)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
2     -0.173444 -0.071659       1        1  24.972587
5     -0.098807 -0.002706       2        1  17.994203
4     -0.134537 -0.101396       3        1  17.704325
3      0.015769  0.141748       4        1  13.247767
1      0.028567 -0.063550       5        1  11.526480
6      0.061627  0.234562       6        1  11.019215
0      0.300825 -0.137000       7        1   3.535423, topic_info=         Term          Freq         Total Category  logprob  loglift
68       says  45786.000000  45786.000000  Default  30.0000  30.0000
80      trump  23031.000000  23031.000000  Default  29.0000  29.0000
65       said  28814.000000  28814.000000  Default  28.0000  28.0000
69      state  10582.000000  10582.000000  Default  27.0000  27.0000
58    percent   8053.000000   8053.000000  Default  26.0000  26.0000
..        ...           ...           ...      ...      ...      ...
29      going    192.204536   9424.960818   Topic7  -4.8679  -0.5502
55        npr    133.737470   6613.931497   Topic7  -5.2306  -0.5587
61  president    217.697152  11639.094943   Topic7  -4.7434  -0.6367
38       just    240.557135  18832.669399   Topic7  -4.6435  -1.0181
87        way    127.025936  10234.790047   Topic7  -5.2821  -1.0468

[280 rows x 6 columns], token_table=      Topic      Freq   Term
term                        
0         3  0.215967    000
0         4  0.168238    000
0         5  0.609864    000
0         7  0.006035    000
1         1  0.042376     10
...     ...       ...    ...
95        3  0.186341  years
95        4  0.086592  years
95        5  0.211661  years
95        6  0.012316  years
95        7  0.007343  years

[593 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[3, 6, 5, 4, 2, 7, 1])

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


pyLDAvis reference paper: https://cran.r-project.org/web/packages/LDAvis/vignettes/details.pdf
relevance definition: https://nlp.stanford.edu/events/illvi2014/papers/sievert-illvi2014.pdf